# Test a common residual subspace across rules
Fit on SOURCES only; choose rank on their selection pairs only; evaluate TARGETS. Include a rule in TARGETS but exclude it from SOURCES to test transfer of the basis to that rule. Its discovery mean calibrates the intervention center, and its own-rule control uses its discovery contrasts; neither enters shared-basis fitting or rank selection.

`consensus` averages projectors onto each source's top eight contrast directions. `pooled` fits SVD to equally weighted per-example source contrasts. `mean` reproduces rank-one SVD of normalized rule means (RANKS=[1]). A pooled success may combine separate mechanisms rather than identify a common one. If no candidate meets all source thresholds the notebook explicitly marks its choice diagnostic. Those thresholds are screening rules, not significance tests.

All fitted bases, split records, margins, base controls, own controls and five random controls are exported. The 3D mean-direction graph uses origin-preserving uncentered SVD/PCA and reports retained energy and full-space cosines. It does not depict the entire higher-rank intervention. Free-generation outputs require separate interpretation. Preservation of ordinary semantic reasoning is NOT established by fixed-pair base controls alone; inspect free-generation accuracy, and add a matched clean-SFT control before making that paper claim.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate pyyaml tqdm matplotlib plotly
from google.colab import files
from pathlib import Path
import os,sys,zipfile,json,hashlib
ROOT=Path('/content/stegano_experiments'); ROOT.mkdir(exist_ok=True)
print('Upload stegano_experiments_bundle.zip')
uploaded=files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as z:
    for name in z.namelist(): assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
for name,digest in json.loads(Path('bundle_manifest.json').read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest,name
from experiments.data import config
from experiments.colab import ensure_adapters,download
cfg=config()


In [ ]:
SOURCES=['s1','voice','clause']
TARGETS=['s1','voice','clause']  # add 'lexical' after training to test held-out-channel transfer
LAYER=18
METHOD='consensus'  # consensus, pooled, or mean
RANKS=[1,2,4,8]  # use [1] for mean
RUN_FREE_GENERATION=True
OUT=Path('residual_generalization_v2')
ensure_adapters(cfg,list(dict.fromkeys(SOURCES+TARGETS)))


In [ ]:
from experiments.residual import transfer
selected=transfer(cfg,SOURCES,TARGETS,LAYER,OUT,ranks=RANKS,method=METHOD)
download(OUT)


In [ ]:
from experiments.residual import free_generation
if RUN_FREE_GENERATION:
    for rule,intervention in selected.items():
        free_generation(cfg,intervention,OUT/rule/'free')
    download(OUT)
